In [1]:
import polars as pl
import pandas as pd
import json

import sys

sys.path.append("/abyss/home/citation-recommendation/utils")
from API_request import APIRequest as ar

In [14]:
import json

with open('finallength.json') as data_file:    
    data = json.load(data_file)

contextList = []
testList = []
for d in data:
    citedPID = d["citedPaper"]
    citingPID = d["citingPaper"]
    sent = d["context"][0]["citedSent"].replace("<REF>","TARGETCIT")

    obj = {
        str(citedPID) + "_" + str(citingPID) : {
            "masked_text": sent,
            "context_id": str(citedPID) + "_" + str(citingPID),
            "citing_id": str(citedPID),
            "refid": str(citingPID)
            
        }
    }
    contextList.append(obj)
    testList.append({"context_id": str(citedPID) + "_" + str(citingPID), "positive_ids": [str(citingPID)]})

with open('contextlength.json', 'w') as f:
    json.dump(contextList, f)  
    
with open('testlength.json', 'w') as f:
    json.dump(testList, f)  





In [4]:
title1 = [t["title"] for t in data["citedPaperMetaInfo"].to_list()]
title2 = [t["title"] for t in data["citingPaperMetaInfo"].to_list()]
titles = title1 + title2



In [7]:
# for i in range(30):
abstractData = []

for i in range(30):
    df = pl.read_parquet(f"/local/s2orc/{i}.parquet")
    res = df.filter(pl.col("corpusid").is_in(result))
    ids = res['content.source.pdfsha'].to_list()
    
    res = df.filter(pl.col("corpusid").is_in(result))
    # resSelect = res.select(["content.text","content.annotations.abstract"])
    ddd = res.to_pandas()
    dx = pd.DataFrame(ddd)
    indx = dx["content.annotations.abstract"]
    for index,i in enumerate(indx):
        if i is not None:
            idata = json.loads(i)[0]
            abstract = dx["content.text"][index][int(idata["start"]):int(idata["end"])]
            if "citation:" not in abstract.lower():
                abstractData.append(abstract)
            else:
                abstractData.append(None)
        else:
            data = ar.getMetaInfo(dx['content.source.pdfsha'][index])
            if data["abstract"]:
                abstractData.append(data["abstract"])
            else:
                abstractData.append(None)



In [8]:
finalRes = []
for i in range(len(abstractData)):
    if abstractData[i] is not None:
        obj = {result[i]: {"title": titles[i], "abstract": abstractData[i]}}
        finalRes.append(obj)

with open('papers1.json', 'w') as f:
    json.dump(finalRes, f)

In [23]:
# # for obj in finalRes:
# #     # Get the single key in the JSON object
# key_list = list(finalRes.keys())

# # Check if there is a key before using it
# if key_list:
#     first_key = key_list[0]
#     print("The key is:", first_key)
# else:
#     print("No key found in the JSON object.")
keys = []
for r in finalRes:
    first_key = next(iter(r))
    keys.append(first_key)
    
    




In [29]:
# Parse the JSON array
with open("length.json", 'r') as file:
    json_array = json.load(file)
final = []
# Loop through the JSON array
for item in json_array:
    if item["citedPaper"] in keys or item["citedPaper"] in keys:
        final.append(item)
        
        
        
# # Specify the path for the output JSON file
# output_file_path = 'finalLength.json'

# # Write the JSON array to the file
with open('finalLength.json', 'w') as output_file:
    json.dump(final, output_file, indent=2) 
        
        

In [ ]:
with open("length.json", 'r') as file:
    json_array = json.load(file)
    
